# 02 - Modeling
Alur: bandingkan model (CV Macro F1) -> tuning model terbaik -> CV final + bobot kelas -> evaluasi -> feature importance -> simpan model.

In [ ]:
import sys
import json
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

sys.path.insert(0, "..")      

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

from scripts.preprocess import load_data, split_xy
from scripts.train import (
    MODEL_NAMES, build_pipeline, cross_validate_macro_f1,
    optimize_class_weights, save_pipeline, subsample, tune_pipeline,
)
from scripts.visualize import (
    plot_actual_vs_predicted, plot_confusion_matrix, plot_feature_importance,
    plot_model_comparison, plot_per_class_metrics,
)

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA_DIR = next(Path("/kaggle/input").glob("**/train.csv")).parent
    OUT_DIR = Path("/kaggle/working")
else:                                   # lokal: csv di ../data, output ke root repo
    DATA_DIR = Path("../data")
    OUT_DIR = Path("..")
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR)

# ---- parameter eksperimen ----
SEED = 42
SAMPLE_N = 100_000    # subset untuk membandingkan model dan tuning (agar cepat)
TUNE_ITER = 12        # jumlah kombinasi RandomizedSearchCV
CV_SPLITS = 5         # fold CV final pada seluruh data

## 1. Memuat data

In [ ]:
train, test, sample = load_data(DATA_DIR)
print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)
X, y = split_xy(train)
train.head()

## 2. Perbandingan model (subset data, 3-fold)

In [ ]:
X_s, y_s = subsample(X, y, SAMPLE_N, SEED)
rows = []
for name in MODEL_NAMES:
    print(name)
    res = cross_validate_macro_f1(build_pipeline(name, SEED), X_s, y_s, n_splits=3, seed=SEED)
    rows.append({"model": name, "macro_f1_mean": res["mean"], "macro_f1_std": res["std"]})
comparison = pd.DataFrame(rows).set_index("model").sort_values("macro_f1_mean", ascending=False)
BEST_MODEL = comparison.index[0]
print("Model terbaik:", BEST_MODEL)
plot_model_comparison(comparison)
comparison

## 3. Tuning model terbaik (`RandomizedSearchCV`, di dalam pipeline)

In [ ]:
tuned, best_params, tune_score = tune_pipeline(
    build_pipeline(BEST_MODEL, SEED), X_s, y_s, BEST_MODEL, n_iter=TUNE_ITER, seed=SEED
)
print("Param terbaik:", best_params)
print(f"Macro F1 CV (tuning): {tune_score:.4f}")

## 4. CV pada seluruh data + optimasi bobot kelas

In [ ]:
cv = cross_validate_macro_f1(tuned, X, y, n_splits=CV_SPLITS, seed=SEED)
print(f"Macro F1 CV {CV_SPLITS}-fold: {cv['mean']:.4f} +/- {cv['std']:.4f}")

weights, weighted_score = optimize_class_weights(cv["oof_proba"], y, cv["classes"])
print("Bobot kelas:", np.round(weights, 2).tolist())
print(f"Macro F1 OOF dengan bobot: {weighted_score:.4f}")

pred_oof = cv["classes"][(cv["oof_proba"] * weights).argmax(axis=1)]
print(classification_report(y, pred_oof, digits=3))

## 5. Evaluasi: confusion matrix, metrik per kelas, distribusi prediksi

In [ ]:
plot_confusion_matrix(y, pred_oof)
plot_per_class_metrics(y, pred_oof)
plot_actual_vs_predicted(y, pred_oof)

## 6. Feature importance (permutation importance pada hold-out)

In [ ]:
imp = plot_feature_importance(tuned, X_s, y_s, seed=SEED)
imp.sort_values(ascending=False).head(10)

## 7. Fit akhir dan simpan model

In [ ]:
final_model = tuned.fit(X, y)
model_path = save_pipeline(final_model, BEST_MODEL, weights)
print("Model tersimpan:", model_path)

REPORT_DIR = OUT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "model": BEST_MODEL,
    "best_params": {k: (v.item() if hasattr(v, "item") else v) for k, v in best_params.items()},
    "cv_folds": CV_SPLITS,
    "cv_macro_f1_mean": cv["mean"],
    "cv_macro_f1_std": cv["std"],
    "macro_f1_oof_with_class_weights": weighted_score,
    "class_weights": [float(w) for w in weights],
}
(REPORT_DIR / f"cv_{BEST_MODEL}.json").write_text(json.dumps(summary, indent=2))
summary

# 8. Interpretasi

## Model Terbaik dan Alasannya
Model terbaik adalah **Regresi Logistik (`reglog`)**.

**Alasan:**
- Pada perbandingan awal, reglog memiliki `macro_f1_mean` tertinggi = **0.500636** dan std terkecil = **0.001305**.
- Lebih baik dan lebih stabil dibanding:
  - HGB: 0.485054 ± 0.003285
  - RF: 0.482402 ± 0.002594
- Setelah tuning, parameter terbaiknya adalah **C ≈ 2.5378**.
- Setelah ditambah bobot kelas, **Macro F1 OOF = 0.5224**.

**Model final yang dipilih:**
- Logistic Regression dengan `C ≈ 2.538`
- Bobot kelas: `[0.5, 0.8, 1.0, 1.1, 1.2]`

---

## Pengaruh Tuning dan Bobot Kelas terhadap Macro F1

### Tuning Hyperparameter C
- Dilakukan **3-fold** untuk **12 kandidat**, total **36 fit**.
- Parameter terbaik: **C ≈ 2.5378**
- Skor tuning 3-fold: **0.5007**
- Evaluasi 5-fold: **0.5158 ± 0.0019**
- Dibanding baseline reglog 0.5006, tuning cenderung memberi perbaikan tipis/sedang dan tetap stabil karena std kecil.

### Bobot Kelas
- Bobot yang diterapkan: `[0.5, 0.8, 1.0, 1.1, 1.2]`
- **Macro F1 OOF naik menjadi 0.5224**
- Kenaikan sekitar **+0.0066** dari CV 5-fold 0.5158.
- Ini menunjukkan pembobotan kelas membantu model lebih seimbang secara agregat macro, meskipun kelas 2 dan 3 tetap lemah.

---

## Kelas yang Paling Sulit Diprediksi dan Pola Kesalahan

### Kelas Paling Sulit
Kelas **2** adalah yang paling sulit diprediksi:
- Precision: **0.347**
- Recall: **0.406**
- F1: **0.374**

Sebaliknya, kelas **5** paling mudah:
- Precision: **0.753**
- Recall: **0.677**
- F1: **0.713**

### Pola Kesalahan
- Kelas tengah, terutama **2 dan 3**, paling sulit dibedakan.
- Precision rendah pada kelas 2 dan 3 berarti banyak prediksi **false positive**.
- Recall rendah berarti banyak sampel kelas 2/3 justru diprediksi sebagai kelas lain.
- Indikasinya kelas 2 dan 3 sering tertukar dengan kelas di sekitarnya, misalnya kelas 1–4.
- Pola ini merupakan ciri khas **klasifikasi ordinal** (kesalahan terjadi pada kelas tetangga).

---

## Fitur Paling Berpengaruh dan Penjelasan Rekayasanya

### Top 3 Fitur Paling Berpengaruh
| Fitur | Importance |
|---|---:|
| `district_id` | 0.107532 |
| `ward_id` | 0.103773 |
| `vdcmun_id` | 0.068662 |

### Penjelasan Struktural
- `district_id`, `ward_id`, `vdcmun_id` adalah fitur **lokasi administratif**.
- Secara struktural tidak langsung, tetapi masuk akal sebagai **proxy kondisi wilayah**:
  - Praktik konstruksi lokal
  - Kode bangunan
  - Tingkat bahaya gempa
  - Jenis tanah
  - Sosial-ekonomi
  - Kualitas infrastruktur
- Perlu hati-hati karena bisa menimbulkan **leakage/overfit spasial** jika model dipakai ke wilayah baru.
